# 04 - Scene inference and monitoring time series
Run the model over many dates on the same grid, export probability rasters, and compute hyacinth area and cover per date and per month.

Interpretation caveats: cover is computed only over *observed* (cloud-free) water; months with little visible water are dropped by `monthly()`.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from hyacinthwatch.config import load_config
cfg = load_config("../config/config.yaml")
bbox = cfg["aoi"]["pilot_bbox"]
BANDS = cfg["bands"]
import torch, rioxarray as rxr
from pathlib import Path
from hyacinthwatch.stac_data import search_scenes, load_scene
from hyacinthwatch.weak_labels import valid_mask
from hyacinthwatch.model import build_model
from hyacinthwatch.inference import predict_scene, save_raster
from hyacinthwatch.timeseries import scene_stats, monthly

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
st = np.load("../models/stats.npz"); mean, std = st["mean"], st["std"]
t = cfg["training"]
model = build_model(len(BANDS), t["encoder"], weights=None).to(device)
model.load_state_dict(torch.load("../models/unet.pt", map_location=device))
water_da = rxr.open_rasterio("../data/outputs/water_mask.tif").squeeze("band", drop=True)
water = water_da.values.astype(bool)

In [ ]:
items = search_scenes(bbox, cfg["stac"]["date_range"], cfg["stac"]["max_cloud"])
Path("../data/outputs/prob").mkdir(parents=True, exist_ok=True)
rows = []
for it in items:
    try:
        stack, scl = load_scene(it, bbox, BANDS, match=water_da)
        prob = predict_scene(model, stack.values, mean, std, device)
        valid = valid_mask(scl.values)
        rows.append(scene_stats(it.datetime, prob, water, valid, t["threshold"]))
        save_raster(prob, stack, f"../data/outputs/prob/{it.id}.tif")
    except Exception as e:
        print("skip", it.id, e)
ts = pd.DataFrame(rows).sort_values("date")
ts.to_csv("../data/outputs/timeseries.csv", index=False)
ts.tail()

In [ ]:
m = monthly(ts)
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(ts.date, ts.cover_pct, ".", alpha=0.4, label="per scene")
ax.plot(m.index, m.cover_pct, "-o", label="monthly median")
ax.set_ylabel("hyacinth cover of observed water (%)"); ax.legend(); plt.show()